### Q1: Build Your Personalized Knowledge Base
Roll Number: 1024170374
Last two digits: 74. 
7 % 3 = 1 ('account')
4 % 3 = 1 ('account')
Building a pandas DataFrame with 4 fixed entries and 2 personalized entries.

In [1]:
import pandas as pd

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"}
]

# Roll number: 1024170374 -> last two digits: 74
# 7 % 3 = 1 -> 'account'
# 4 % 3 = 1 -> 'account'
personalized_entries = [
    {"question": "how do I change my email address", "answer": "Go to account settings and update your email.", "keywords": "email address update account", "category": "account"},
    {"question": "how to enable two factor authentication", "answer": "Navigate to security settings and enable 2FA.", "keywords": "security 2fa authentication account", "category": "account"}
]

all_entries = fixed_entries + personalized_entries
df = pd.DataFrame(all_entries)
print(df)

                                     question                                     answer                          keywords  category
0                         what is the annual fee              The annual fee is Rs 500.             fee cost price charge   billing
1                         how to reset password            Go to Settings > Reset Password.               password reset login   account
2                     what are your working hours                 We are open 9 AM to 5 PM.           hours timing open time   general
3                         how can i pay the fee  You can pay via UPI, card, or net banking.            pay payment upi fee   billing
4             how do I change my email address   Go to account settings and update your email.      email address update account   account
5  how to enable two factor authentication       Navigate to security settings and enable 2FA.  security 2fa authentication account   account


### Q2: Generate and Score a Hypothesis
Implement a scoring function that takes a query string and returns all matching entries ranked by confidence.

In [2]:
def score_query(query, dataframe):
    query_tokens = set(query.lower().split())
    scores = []
    for idx, row in dataframe.iterrows():
        # Combine question and keywords for matching
        entry_text = (row['question'] + " " + row['keywords']).lower()
        entry_tokens = set(entry_text.split())
        match_count = len(query_tokens.intersection(entry_tokens))
        scores.append(match_count)
    
    df_scored = dataframe.copy()
    df_scored['score'] = scores
    # Filter out zero scores and sort by score descending
    df_ranked = df_scored[df_scored['score'] > 0].sort_values(by='score', ascending=False)
    return df_ranked

query_str = "fee payment"
print(f"Query: '{query_str}'")
print(score_query(query_str, df))

Query: 'fee payment'
                                     question                                     answer               keywords  category  score
3                       how can i pay the fee  You can pay via UPI, card, or net banking.    pay payment upi fee   billing      2
0                      what is the annual fee              The annual fee is Rs 500.  fee cost price charge   billing      1


### Q3: Category Filter Function
Write a function `same_category(category_name, df)` that returns all questions belonging to a given category.

In [3]:
def same_category(category_name, dataframe):
    filtered = dataframe[dataframe['category'] == category_name]
    return filtered['question']

# Call using category of one of the personalized entries ('account')
cat_to_check = "account"
print(f"Category: '{cat_to_check}'")
print(same_category(cat_to_check, df))

Category: 'account'
1                       how to reset password
4            how do I change my email address
5  how to enable two factor authentication
Name: question, dtype: object


### Q4: Update Keywords and Save to CSV
Pick any one entry, add a new keyword, and save the updated DataFrame to `1024170374_faq_data.csv`.

In [4]:
# Picking entry 0 and adding a new keyword 'charges'
new_keyword = "charges"
current_keywords = df.loc[0, 'keywords']
if new_keyword not in current_keywords:
    df.loc[0, 'keywords'] = current_keywords + " " + new_keyword

print(f"Updated entry 0 keywords: {df.loc[0, 'keywords']}")

filename = "1024170374_faq_data.csv"
df.to_csv(filename, index=False)
print(f"Successfully saved DataFrame to {filename}")

Updated entry 0 keywords: fee cost price charge charges
Successfully saved DataFrame to 1024170374_faq_data.csv


### Q5: Groupby Category Count
Using groupby, print how many FAQ entries you have per category.

In [5]:
category_counts = df.groupby('category')['question'].count()
print(category_counts)

category
account    3
billing    2
general    1
Name: question, dtype: int64


### Q6: Tie-Handling Scoring Function
Modify scoring function to handle ties by printing all matching entries with the highest score. Demonstrate with one query that produces a tie and one that doesn't.

In [6]:
def score_query_with_ties(query, dataframe):
    query_tokens = set(query.lower().split())
    scores = []
    for idx, row in dataframe.iterrows():
        entry_text = (row['question'] + " " + row['keywords']).lower()
        entry_tokens = set(entry_text.split())
        match_count = len(query_tokens.intersection(entry_tokens))
        scores.append(match_count)
    
    df_scored = dataframe.copy()
    df_scored['score'] = scores
    df_filtered = df_scored[df_scored['score'] > 0]
    
    if df_filtered.empty:
        print("No matching entries found.")
        return
    
    max_score = df_filtered['score'].max()
    top_matches = df_filtered[df_filtered['score'] == max_score]
    
    if len(top_matches) > 1:
        print(f"Found tie for highest score ({max_score}):")
    else:
        print(f"Highest score entries:")
        
    print(top_matches)

print("--- Demonstration 1: Query producing a tie ('fee') ---")
score_query_with_ties("fee", df)

print("\n--- Demonstration 2: Query without a tie ('working') ---")
score_query_with_ties("working", df)

--- Demonstration 1: Query producing a tie ('fee') ---
Query: 'fee'
Found tie for highest score (1):
                                  question                             answer               keywords  category  score
0                   what is the annual fee          The annual fee is Rs 500.  fee cost price charge  billing      1
3                    how can i pay the fee  You can pay via UPI, card, or net banking.    pay payment upi fee  billing      1

--- Demonstration 2: Query without a tie ('working') ---
Query: 'working'
Highest score entries:
                      question                     answer                keywords  category  score
2  what are your working hours  We are open 9 AM to 5 PM.  hours timing open time   general      1
